# Gaussian Parcels · Bilateral Constriction · GPU

**همان مدل قبلیِ سیال–ابر ذرات گاوسی، با افزودن دو گرفتگی نیم‌دایره‌ای.**
حل زمانی دوبعدی، دو شبکهٔ سیال و پارسل، درگ استوکس، کوپل دوطرفهٔ گاوسی، مرز تناوبی x و نیروی محرک قبلی حفظ شده‌اند.
این نسخه پایا یا بدون ذره نیست. طول L/H=3، Re=20، تعداد پارسل، عرض فیلتر، گرانش و چهار حالت استوکس/بارگذاری همان‌اند.
در Colab، GPU را فعال و Run all کنید؛ کد و داده‌های تحلیلی لازم داخل همین notebook هستند.
هدف ۵۵ دقیقه و حداکثر ۳۵ دقیقه آموزش است؛ بودجهٔ زمان تضمین دقت نیست.

شرط اولیهٔ سیال برای رعایت مرز خمیده سازگار می‌شود؛ در شعاع صفر مدل قبلی دقیقاً بازیابی می‌شود.
جامد از نقاط حل و انتگرال کرنل حذف می‌شود. مدل همچنان رقیق و بدون برخورد است؛ ورود مرکز به جامد با loss جریمه و صریح گزارش می‌شود.
مسیر یا برخورد ساختگی به خروجی اضافه نمی‌شود. اعتبارسنجی مقاله‌ای هندسهٔ جدید ادعا نمی‌شود.

In [ ]:
import time, os
from pathlib import Path
from datetime import datetime, timezone
import subprocess, sys, json

NOTEBOOK_STARTED = time.monotonic()
TOTAL_MINUTES = 55       # سقف کل؛ بیشتر از 55 مجاز نیست
TRAIN_MINUTES = 35       # سهم کل آموزش؛ بین چهار حالت تقسیم می‌شود
PRECISION = 'float32'    # روی T4 مناسب‌تر؛ float64 کندتر است
OUTPUT_BASE = Path.cwd() / 'gaussian_constricted_results'
# برای ماندگاری، می‌توانید OUTPUT_BASE را به مسیر Drive متصل‌شده تغییر دهید.
assert 0 < TOTAL_MINUTES <= 55 and TRAIN_MINUTES > 0
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
WORK = Path.cwd() / f'gaussian_constricted_source_{RUN_ID}'
OUT = OUTPUT_BASE / f'run_{RUN_ID}'
NOTEBOOK_DEADLINE = NOTEBOOK_STARTED + TOTAL_MINUTES * 60
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
print('Output:', OUT)

# فقط هندسهٔ افزوده‌شده؛ بقیهٔ فیزیک همان مدل قبلی است
GEOMETRY = {'radius': 0.25, 'center_fraction': 0.4}

## کد همراه notebook
سلول بعدی بستهٔ کد را با SHA-256 بررسی و در پوشه‌ای جدید استخراج می‌کند. فایل‌های قبلی بازنویسی نمی‌شوند.

In [ ]:
import base64, hashlib, io, zipfile
SOURCE_B64 = '...'
SOURCE_SHA256 = 'f1969507955ed48b5261ef6e7d2c42b475334c5c31d86612a69e597a06621aca'
payload = base64.b64decode(SOURCE_B64)
assert hashlib.sha256(payload).hexdigest() == SOURCE_SHA256, 'Source checksum mismatch'
WORK.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert archive.testzip() is None
    for item in archive.infolist():
        assert (WORK / item.filename).resolve().is_relative_to(WORK.resolve())
    archive.extractall(WORK)
print('Source verified:', SOURCE_SHA256)

## آماده‌سازی GPU
PyTorch موجود در محیط حفظ می‌شود تا نسخهٔ CUDA با نسخهٔ CPU جایگزین نشود.
اگر GPU در دسترس نباشد، این notebook صریحاً متوقف می‌شود؛ آموزش مخفیانه روی CPU ادامه پیدا نمی‌کند.

In [ ]:
import importlib.util, importlib.metadata
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU را از تنظیمات runtime فعال کنید و دوباره Run all بزنید.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2, 2)

# Colab/Kaggle معمولاً این بسته‌ها را دارند؛ فقط وابستگی ناسازگار/غایب نصب می‌شود.
constraints = {
    'numpy': ('numpy>=1.26,<3', (1, 26), (3, 0)),
    'scipy': ('scipy>=1.12,<2', (1, 12), (2, 0)),
    'pandas': ('pandas>=2.2,<3', (2, 2), (3, 0)),
    'matplotlib': ('matplotlib>=3.8,<4', (3, 8), (4, 0)),
    'Pillow': ('Pillow>=10,<13', (10, 0), (13, 0)),
    'psutil': ('psutil>=5.9,<8', (5, 9), (8, 0)),
}
needed = []
for distribution, (requirement, lower, upper) in constraints.items():
    try:
        version = tuple(int(p) for p in importlib.metadata.version(distribution).split('.')[:2])
        if not lower <= version < upper:
            needed.append(requirement)
    except (importlib.metadata.PackageNotFoundError, ValueError):
        needed.append(requirement)
if needed:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *needed],
                   check=True, timeout=min(240, max(1, NOTEBOOK_DEADLINE-time.monotonic()-120)))
print('Dependencies ready. Elapsed:', round((time.monotonic()-NOTEBOOK_STARTED)/60, 2), 'min')

## هندسهٔ افزوده‌شده
L/H=3 همان مدل قبلی؛ دو نیم‌دایره با r/H=.25 و x₀/L=.4. مرز x تناوبی و نیروی محرک ثابت مانده‌اند.

In [ ]:
sys.path.insert(0, str(WORK))
from gaussian_constriction.model import Geometry, GeometrySettings
from gaussian_constriction.figures import domain
from parcel_pinn.config import Config
import matplotlib.pyplot as plt
from IPython.display import display, Image
import io
geometry = Geometry(Config().lengths[0], GeometrySettings(**GEOMETRY))
fig, ax = plt.subplots(figsize=(10, 4)); domain(ax, geometry)
ax.set(xlim=(0, geometry.length), ylim=(0, 1), aspect='equal', xlabel='x / H', ylabel='y / H', title='Channel geometry')
buffer = io.BytesIO(); fig.savefig(buffer, format='png', dpi=110, bbox_inches='tight')
display(Image(data=buffer.getvalue())); plt.close(fig)
print('Transient fluid + two-way Gaussian parcels; periodic x; same forcing')

## آموزش و تولید خروجی
چهار حالت شامل اینرسی کم، حالت پایه، بارگذاری بیشتر و اینرسی بیشتر است.
هر حالت شبکه‌های مستقل سیال و مسیر پارسل دارد؛ دادهٔ مرجع وارد آموزش کانال نمی‌شود.
آموزش روی GPU با Adam انجام می‌شود؛ مشتقات معادلات بدون AMP و TF32 محاسبه می‌شوند.
بهترین checkpoint بر اساس batch ثابت نگه داشته می‌شود، حتی اگر loss خراب شود یا بودجهٔ زمان تمام شود.
گزارش و شکل‌ها روی CPU تولید می‌شوند و برای این مرحله زمان جدا رزرو شده است.

In [ ]:
import signal, psutil
remaining_seconds = NOTEBOOK_DEADLINE - time.monotonic() - 20
if remaining_seconds < 180:
    raise RuntimeError('زمان کافی برای اجرای کامل باقی نمانده؛ runtime را آماده و notebook را دوباره اجرا کنید.')
geometry_path = WORK / 'geometry_parameters.json'
geometry_path.write_text(json.dumps(GEOMETRY))
command = [sys.executable, '-u', str(WORK / 'run_constriction.py'),
           '--out', str(OUT), '--geometry', str(geometry_path), '--validation-pack', str(WORK / 'reference_inputs/Validation_Pack'),
           '--device', 'cuda', '--precision', PRECISION,
           '--max-minutes', str(min(55, remaining_seconds/60)),
           '--train-minutes', str(TRAIN_MINUTES)]
# نمایش زندهٔ وضعیت؛ ناظر مستقل کل پردازش را پیش از پایان بودجه متوقف می‌کند.
import threading
proc = subprocess.Popen(command, cwd=WORK, env=os.environ.copy(),
                        start_new_session=os.name == 'posix',
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
def show_progress():
    for line in proc.stdout:
        print(line, end='', flush=True)
progress = threading.Thread(target=show_progress, daemon=True)
progress.start()
hard_timeout = threading.Event()
def stop_all_workers():
    if proc.poll() is None:
        hard_timeout.set()
        # کارگرها session مستقل دارند؛ psutil در Colab/Kaggle معمولاً نصب است.
        # مسیر عادی توسط supervisor داخلی هر worker را به‌صورت مستقل محدود می‌کند.
        try:
            import psutil
            for child in psutil.Process(proc.pid).children(recursive=True):
                try: child.kill()
                except psutil.NoSuchProcess: pass
        except (ImportError, psutil.NoSuchProcess):
            pass
        if os.name == 'posix':
            try: os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError: pass
        else:
            proc.kill()
watchdog = threading.Timer(remaining_seconds + 5, stop_all_workers)
watchdog.daemon = True
watchdog.start()
try:
    returncode = proc.wait()
except KeyboardInterrupt:
    stop_all_workers()
    returncode = proc.wait(timeout=5)
    print('اجرای دستی متوقف شد؛ فایل‌های ساخته‌شده حفظ شده‌اند.')
finally:
    watchdog.cancel()
    progress.join(timeout=2)
print('Exit code:', returncode, '| Elapsed:', round((time.monotonic()-NOTEBOOK_STARTED)/60, 2), 'min')
print('Hard timeout:', hard_timeout.is_set())
print('خروج غیرصفر یعنی یک مرحله ناقص/خطادار بوده؛ جزئیات را در گزارش ببینید.')

## معیارها و جدول نتایج
پایان اجرا به معنای اعتبار علمی نتیجه نیست؛ خطای باقیماندهٔ معادلات با درصد اختلاف با مقاله متفاوت است.

In [ ]:
from IPython.display import display, HTML, Image, FileLink
import pandas as pd
if (OUT / 'summary.json').is_file():
    summary = json.loads((OUT / 'summary.json').read_text())
    print('Workflow complete:', summary['workflow_complete'])
    print('Screening criteria:', summary['research_quality_criteria_met'])
    print('Numerical verification:', summary['verification_passed'], '/', summary['verification_total'])
    print('Literature validation:', summary['physical_validation_cases_run'])
    print('Plot errors:', summary['plot_errors'])
    metrics = OUT / 'paper_metrics.csv'
    if metrics.is_file() and metrics.stat().st_size > 2:
        display(pd.read_csv(metrics))
    display(pd.DataFrame(summary['verification'])[['case_id', 'outcome', 'details']])
else:
    print('گزارش نهایی ساخته نشده است؛ checkpointها و logهای پوشهٔ خروجی را بررسی کنید:', OUT)
if (OUT / 'summary.json').is_file():
    print('Particles:', summary['particles_modeled'], '| physical time:', summary['physical_time_modeled'], '| two-way Gaussian:', summary['two_way_gaussian_coupling'])
    for c in summary['cases']:
        print(c['case'], 'solid centers:', c['max_obstacle_inside_count'], 'kernel error:', c['max_kernel_error'])

## شکل‌ها
همهٔ شکل‌های ثابت از manifest نمایش داده می‌شوند؛ فایل‌های PNG، PDF و SVG در ZIP هستند. نمای عمق‌دار فقط نمایش حل دوبعدی است.

In [ ]:
import html
manifest = OUT / 'figure_manifest.json'
if manifest.is_file():
    for figure in json.loads(manifest.read_text()):
        path = OUT / figure['files'][0]
        if path.suffix == '.png' and path.is_file():
            display(HTML('<h4>' + html.escape(figure['title']) + '</h4>'))
            display(Image(filename=str(path), width=900))
            display(HTML('<p>' + html.escape(figure['caption']) + '</p>'))

## حرکت پارسل‌ها و ابرهای گاوسی
frameها زمان فیزیکی پیش‌بینی‌شده هستند. نفوذ احتمالی مرکز به جامد در گزارش و با دورخط قرمز در انیمیشن مشخص می‌شود؛ موقعیت‌ها clip یا project نمی‌شوند.

In [ ]:
for folder in sorted((OUT / 'cases').glob('*')):
    for name in ['transport.gif', 'gaussian_transport.gif']:
        path = folder / name
        if path.is_file():
            print(folder.name, name)
            display(Image(filename=str(path)))

## دریافت خروجی کامل
ZIP شامل شکل‌ها، انیمیشن‌ها، CSV، میدان‌ها، وزن شبکه‌ها، تنظیمات، logها و کد دقیق اجرای شماست.
`index.html` گالری کامل آفلاین و `REPORT.md` گزارش است. `completion.json` زمان و مراحل ناقص را ثبت می‌کند.
آزمون‌های V01–V10 بررسی عددی اجزای مدل‌اند؛ دادهٔ مقاله‌ای نامرتبط به‌عنوان validation این مدل گزارش نمی‌شود.

In [ ]:
archive = OUT.parent / f'{OUT.name}_results.zip'
if archive.is_file():
    display(FileLink(str(archive)))
    print('ZIP:', archive)
    try:
        from google.colab import files
        files.download(str(archive))
    except ImportError:
        print('در Jupyter/Kaggle فایل ZIP را از پنل فایل‌ها دریافت کنید.')
    except Exception as exc:
        print('دانلود خودکار انجام نشد؛ ZIP را از پنل فایل‌ها بگیرید:', exc)
if (OUT / 'completion.json').is_file():
    print(json.dumps(json.loads((OUT / 'completion.json').read_text()), ensure_ascii=False, indent=2))